In [1]:
import requests
import sqlite3
import pandas as pd

In [2]:
URL = "https://openlibrary.org/search.json?q=python+programming&limit=20&fields=key,title,author_name,first_publish_year"
params = {
    "q": "python programming",
    "limit": 20,
    "fields": "key,title,author_name,first_publish_year",
}

response = requests.get(URL, params=params, timeout=15)
response.raise_for_status()        # stops with an error if the request failed
data = response.json()

books = data["docs"]
print(f"Fetched {len(books)} books")

Fetched 20 books


In [3]:
books[:2]     # look at the first two records

[{'author_name': ['R. Nageswara Rao'],
  'first_publish_year': 2016,
  'key': '/works/OL30906747W',
  'title': 'Core Python Programming'},
 {'author_name': ['Justin Seitz', 'Tim Arnold'],
  'first_publish_year': 2014,
  'key': '/works/OL19547345W',
  'title': 'Black Hat Python'}]

In [4]:
clean_books = []

for b in books:
    title = (b.get("title") or "").strip()
    key = (b.get("key") or "").strip()
    if not title or not key:          # skip unusable records
        continue

    authors = b.get("author_name") or []
    author = ", ".join(authors[:3]) if authors else "Unknown"
    year = b.get("first_publish_year")     # may be None

    clean_books.append((key, title, author, year))

print(f"{len(clean_books)} valid books ready to store")
clean_books[:2]

20 valid books ready to store


[('/works/OL30906747W', 'Core Python Programming', 'R. Nageswara Rao', 2016),
 ('/works/OL19547345W', 'Black Hat Python', 'Justin Seitz, Tim Arnold', 2014)]

In [5]:
conn = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        source_key TEXT NOT NULL UNIQUE,
        title      TEXT NOT NULL,
        author     TEXT NOT NULL,
        pub_year   INTEGER
    )
""")
conn.commit()
print("Table ready")

Table ready


In [6]:
cursor.executemany("""
    INSERT INTO books (source_key, title, author, pub_year)
    VALUES (?, ?, ?, ?)
    ON CONFLICT(source_key) DO UPDATE SET
        title    = excluded.title,
        author   = excluded.author,
        pub_year = excluded.pub_year
""", clean_books)

conn.commit()
print("Books saved to database")

Books saved to database


In [7]:
df = pd.read_sql_query(
    "SELECT title, author, pub_year FROM books ORDER BY pub_year, title",
    conn,
)
df

,title,author,pub_year
0,Programming Python,Mark Lutz,1996
1,Learning Python,"Mark Lutz, David Ascher",1999
2,Python programming,John M. Zelle,2003
3,Core Python programming,Wesley Chun,2006
4,Python programming in context,Bradley N. Miller,2009
5,Black Hat Python,"Justin Seitz, Tim Arnold",2014
6,Python Programming For Beginners,"Python Programming, Michael Knapp",2015
7,Core Python Programming,R. Nageswara Rao,2016
8,Murach's Python Programming,"Michael Urban, Joel Murach",2016
9,Murach's Python Programming (2nd Edition),"Joel Murach, Michael Urban",2016


In [8]:
pd.read_sql_query("SELECT COUNT(*) AS total_books, MIN(pub_year) AS oldest, MAX(pub_year) AS newest FROM books", conn)

,total_books,oldest,newest
0,20,1996,2022


This notebook fetches books from the Open Library API, cleans the data, stores it in a SQLite database, and displays it as a table read back from the database.

Missing fields: Missing authors become "Unknown", missing years stay empty, and records without a title or ID are skipped.

Duplicates: Each book's API ID is a unique key, so re-running the notebook updates existing rows instead of adding copies.

Improvements: I would add pagination to fetch more than 20 books, retry logic for slow or failed requests, and a simple chart of books by decade.